# SBERT Training + Ranking + Evaluation Pipeline

In [10]:
import sys
import os
import json
sys.path.append('.')

from preprocessing.data_loader import load_training_examples, load_dev_data
from training.train import build_model, build_dataloader, train_model, save_model_pt
from evaluation.evaluate import rank_all_queries, evaluate_all

In [11]:
# Load data
train_path = "processed_data/train_pairs.json"
dev_path = "processed_data/dev_candidates.json"

print("Loading data...")
train_data = load_training_examples(train_path)
dev_data = load_dev_data(dev_path)
print(f"Loaded {len(train_data)} training examples and {len(dev_data)} dev queries.")

Loading data...
Loaded 220000 training examples and 3136 dev queries.


In [12]:
# Build and Train Model
print("Building model...")
model = build_model("all-MiniLM-L6-v2")

print("Building dataloader...")
dataloader = build_dataloader(train_data, batch_size=16)

print("Training model...")
train_model(model, dataloader, loss_name="cosine", epochs=1)

print("Saving model...")
save_model_pt(model, "model.pt")

Building model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building dataloader...
Training model...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/Users/Nishanth/Documents/nlp/nlp_proj_final/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.018353
1000,0.016645
1500,0.015534
2000,0.014803
2500,0.014917
3000,0.013979
3500,0.014052
4000,0.014038
4500,0.014327
5000,0.013599


Saving model...


In [13]:
# Ranking and Evaluation
print("Ranking all queries...")
top_k = 10
rankings = rank_all_queries(model, dev_data, top_k=top_k)

print("Evaluating...")
metrics = evaluate_all(rankings, dev_data, k=top_k)

print("\n--- Final Metrics ---")
print(json.dumps(metrics, indent=2))

Ranking all queries...


Ranking Queries:   0%|          | 0/3136 [00:00<?, ?it/s]

Evaluating...

--- Final Metrics ---
{
  "Recall@10": 0.9990433673469388,
  "MRR": 0.9972231079931972,
  "Accuracy@10": 1.0
}
